In [264]:
import pandas as pd
import numpy as np
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

# FUNCTIONS FOR PREPROCESSING FOR TRAIN SET

In [265]:
# Split datasets

def split_sets(X, test_size=0.2, random_state=42):
    X['Date of Birth'] = pd.to_datetime(X['Date of Birth'], errors='coerce') # This is becacuse the DoB cannot be converted in X_test after splitting
    y_new = X['Survival Prediction'].copy()
    X_new = X.drop(columns=['Survival Prediction']).copy()
    X_train, X_test, y_train, y_test = train_test_split(X_new, y_new, test_size=test_size, random_state=random_state)
    y_train.replace({'No': 0, 'Yes': 1}, inplace=True)
    y_test.replace({'No': 0, 'Yes': 1}, inplace=True)
    print(f"Dimension of X_train: {X_train.shape}")
    print(f"Dimension of X_test: {X_test.shape}")
    print(f"Dimension of y_train: {y_train.shape}")
    print(f"Dimension of y_test: {y_test.shape}")
    return X_train, X_test, y_train, y_test

In [266]:
# Remove columns

# According to EDA, columns to remove: Transfusion History, Marital Status, Smoking History (due to high correlation with 'Non Smoker', plus this column has missing values)

def remove_columns(df, columns_to_delete):
  df_new = df.copy()
  for column in columns_to_delete:
    df_new = df_new.drop(columns=[column], errors='ignore')
  return df_new

In [267]:
# Data imputation for categorical variables (with the mode)

# Columns to process here: 'Healthcare Access' and 'Gender'

def df_cat_imputation(df, values_to_imput_cat):
  df_new = df.copy()
  mode_train={}
  columns_not_in_list = [x for x in df_new.select_dtypes(include = 'object').columns if x not in values_to_imput_cat.keys()]

  for column, value_to_imput in values_to_imput_cat.items():
      # Get mode from columns in values_to_imput_cat (removing rows with strange values)
        mode_1 = df_new.loc[df_new[column] != values_to_imput_cat[column], column].mode().dropna()
        if not mode_1.empty:
          mode_train[column] = mode_1[0]
          # Replace missing values
          df_new.loc[df_new[column] == values_to_imput_cat[column], column] = mode_train[column]

  # If column is not in values_to_imput_cat, save its mode as it'll be used to imput missing values in the test set.
  for column in columns_not_in_list:
        mode_2 = df_new[column].mode().dropna()
        if not mode_2.empty:
            mode_train[column] = mode_2[0]

  return df_new, mode_train

In [268]:
# Remove duplicates and rows with null values

def remove_rows(df, y_train):
    # 1. Remove duplicates
    df_no_dups = df.drop_duplicates()
    duplicates_removed = len(df) - len(df_no_dups)
    #print(f"# duplicates removed: {duplicates_removed}")

    # 2. Remove rows with missing values
    df_clean = df_no_dups.dropna()
    rows_removed = len(df_no_dups) - len(df_clean)
    #print(f"Number of rows deleted for having missing values {rows_removed}")

    # Save indices from the clean df
    surviving_indices = df_clean.index

    # Filter y_train with surviving indices
    y_clean = y_train.loc[surviving_indices]

    # Reset indices if needed
    df_clean = df_clean.reset_index(drop=False)
    y_clean = y_clean.reset_index(drop=True)

    return df_clean, y_clean

In [269]:
# Standardize values in 'Urban or Real' column

def standardize_urbal_rural (df):
  df_new = df.copy()
  df_new['Urban or Rural'] = df['Urban or Rural'].str.lower()
  df_new['Urban or Rural'] = df['Urban or Rural'].str.capitalize()

  return df_new

In [270]:
# Encode nominal variables into booleans

# Nominal columns (Yes/No): 'Diabetes History', 'Heart Disease History', 'Inflammatory Bowel Disease',
#        'Survival Prediction', 'Diabetes', 'Alcohol Consumption', 'Early Detection',
#        'Family History', 'Genetic Mutation'

def convert_into_bool(df, binary_cols):
  df_new = df.copy()
  for col in binary_cols:
        if col in df_new.columns:
            df_new[col] = df_new[col].map({'Yes': 1, 'No': 0}).astype(int)

  return df_new

In [271]:
# The following columns: {'Cancer Stage', 'Diet Risk', 'Healthcare Access', 'Obesity BMI', 'Physical Activity', 'Screening History'} are ordinal variables that need to be encoded respecing to their natural order.
# ordinal_mappings argument contains the correct ordering of the values for each column.

def apply_ordinal_encoding(df, ordinal_mappings):
    df_encoded = df.copy()
    encoders = {}
    
    for col, ordered_values in ordinal_mappings.items():
        if col in df_encoded.columns:
            mapping = {val: i + 1 for i, val in enumerate(ordered_values)}
            df_encoded[col] = df_encoded[col].map(lambda x: mapping.get(x, -1) if pd.notna(x) else -1)
            encoders[col] = mapping
    
    return df_encoded, encoders

In [272]:
# Transform 'Date of Birth' column into 'Age' column

def create_age_column(df, reference_date='2025-01-01'):

      df_new = df.copy()
      if 'Date of Birth' in df_new.columns:
        # Create 'Age' Column
        try:
            df_new['Date of Birth'] = pd.to_datetime(df_new['Date of Birth'], errors='coerce')
            ref_date = pd.Timestamp(reference_date)
            df_new['Age'] = ((ref_date - df_new['Date of Birth']).dt.days / 365.25).round()
            df_new['Age'] = df_new['Age'].astype('float')

            # Delete 'Date of birth'
            df_new = df_new.drop(columns=['Date of Birth'])
            #print(f"'Date of Birth' transformed into 'Age'")
        except Exception as e:
            print(f"There was an error {e}")
      return df_new

In [273]:
# Preprocessing for numerical variables (winsorization for outliers, imputation with median)

# Numerical columns to preprocess here: 'Healthcare Costs', 'Incidence Rate per 100K', 'Mortality Rate per 100K',
#        'Tumor Size (mm)', 'Age

# Column 'Healthcare Costs' has negative values. These will be imputated with the median.

# Added the argument 'scaling_true' to scale the variables using the median if needed (0 = no scaling, 1 = scaling).

def df_num_imputation(df, numeric_cols, scaling_true = 1):
  df_new = df.copy()
  stats_pre = {}

# Transform columns into float type
  for col in numeric_cols:
    if col in df_new.columns:
      df_new[col] = pd.to_numeric(df_new[col], errors='coerce')

# Remove outliers using 'Winsorization'
  for column in numeric_cols:
      Q1 = df_new[column].quantile(0.25)
      Q3 = df_new[column].quantile(0.75)
      IQR = Q3 - Q1

      lower_bound = Q1 - 1.5 * IQR
      upper_bound = Q3 + 1.5 * IQR

      # Clip outliers with upper or lower bound
      #df_new[column] = df_new[column].clip(lower=lower_bound, upper=upper_bound) # Uncomment if test fails

      # Imputing negative values with the median
      median_c = df_new[column].median()
      df_new.loc[df_new[column] < 0, column] = median_c

      # BORRAR SI NO FUNCIONA
      min_val = df_new[column].min()
      max_val = df_new[column].max()

      #MANTENER SIEMPRE
      stats_pre[column] = {
          'lower_bound': lower_bound,
          'upper_bound': upper_bound,
          'median': median_c,
          'min': min_val,
          'max': max_val
        }

  # Scaling variables using the median
  if scaling_true == 1:

    #for column in numeric_cols:
    #  if column in stats_pre:
    #     median = stats_pre[column]['median']
    #     df_new[column] = (df_new[column] - median) / median

    #BORRAR SI NO FUNCIONA
    if scaling_true == 1:
      for column in numeric_cols:
        if column in df_new.columns:
          min_val = stats_pre[column]['min']
          max_val = stats_pre[column]['max']
          # Avoid division by zero
          if max_val > min_val:
            # Manual implementation of MinMaxScaler formula: (x - min) / (max - min)
            df_new[column] = (df_new[column] - min_val) / (max_val - min_val)
  
  # Returning the df and the stats_pre dictionary which will be used later to preprocess the test set
  return df_new, stats_pre

In [274]:
# Create dummy columns for categorical ordinal variables

# Those categoricals with less than 2 unique values will be label encoded.

# The main idea is to get rid of categorical columns by turning them into dummies

# Columns to process here
#'Country', 'Gender',
#        'Insurance Costs', 'Insurance Status',
#        'Smoking History', 'Treatment Type', 'Urban or Rural'

def create_dummies(df, categorical_cols):
  df_new = df.copy()
  for col in categorical_cols:
    if col in categorical_cols:
      unique_values = df_new[col].nunique()
      if unique_values > 2:  # One-hot encoding will be performed on variables with more than 2 unique values
        dummies = pd.get_dummies(df_new[col], prefix=col, drop_first=True)
        df_new = pd.concat([df_new, dummies], axis=1)
        df_new = df_new.drop(columns=[col])  # Drop original column after getting dummies

  # Identify remaining categorical columns

  non_numeric_cols = []
  for col in df_new.columns:
      if df_new[col].dtype == 'object':
          #print(f"Non numeric column found: {col}")
          non_numeric_cols.append(col)

  # Transform previous columns

  for col in non_numeric_cols:
    # Customized code for 'Gender' column
    if col in non_numeric_cols:
      if col == 'Gender' and 'Gender' in non_numeric_cols:
        df_new['Gender'] = df_new['Gender'].map({'M': 0, 'F': 1}).astype(bool)
      else:
        dummies = pd.get_dummies(df_new[col], prefix=col, drop_first=True).astype(bool)
        df_new = pd.concat([df_new, dummies], axis=1)
        df_new = df_new.drop(columns=[col])

  # Check for non-numeric remaining columns

  for col in df_new.columns:
    if df_new[col].dtype == 'object':
      print(f"Error: Column {col} remains as non-numeric")

  return df_new

In [275]:
# Macro function for gathering the previous ones

def preprocess_train_df(X, y, columns_to_delete, values_to_imput_cat, binary_cols, ordinal_mappings, numeric_cols , categorical_cols):
  # Create a new feature 'Cardiometabolic_Risk' based on 'Diabetes' and 'Heart Disease History'
  X_new = X.copy()

  # FEATURE ENGINEERING 1:
  # Create a new feature 'Cardiometabolic_Risk' based on 'Diabetes' and 'Heart Disease History'
  X_new['Cardiometabolic_Risk'] = ((X_new['Diabetes'] == 'Yes') | (X_new ['Heart Disease History'] == 'Yes')).astype(int)
  # Interaction between cancer stage and tumor size
  X_new['Cancer_Severity'] = X_new['Cancer Stage'].map({'Localized': 1, 'Regional': 2, 'Metastatic': 3}) * X_new['Tumor Size (mm)']
  # Efectiveness of treatment according to stage
  X_new['Treatment_Effectiveness'] = (X_new['Treatment Type'] != 'Palliative').astype(int) * (4 - X_new['Cancer Stage'].map({'Localized': 1, 'Regional': 2, 'Metastatic': 3}))
  # Early detection with appropriate treatment
  X_new['Early_Detection_Treatment'] = X_new['Early Detection'].map({'Yes': 1, 'No': 0}) * (X_new['Screening History'].map({'Never': 1, 'Irregular': 2, 'Regular': 3})) * (X_new['Treatment Type'] != 'Palliative').astype(int)

  X_clean = remove_columns(X_new, columns_to_delete) # Goes first because there are columns with empty values or that are irrelevant.
  print(f"Dimension of X_clean after: {X_clean.shape}")
  X_clean, mode_train = df_cat_imputation(X_clean, values_to_imput_cat) # Goes second because rows with values to be imputed are removed afterwards
  print(f"Dimension of X_clean after: {X_clean.shape}")
  X_clean, y_train_processed = remove_rows(X_clean, y)
  print(f"Dimension of X_clean after: {X_clean.shape}")
  X_clean = standardize_urbal_rural(X_clean)
  print(f"Dimension of X_clean after: {X_clean.shape}")
  X_clean = convert_into_bool(X_clean, binary_cols)
  print(f"Dimension of X_clean after: {X_clean.shape}")
  X_clean, encoders = apply_ordinal_encoding(X_clean, ordinal_mappings)
  print(f"Dimension of X_clean after: {X_clean.shape}")

  # FEATURE ENGINEERING 2:
  X_clean = create_age_column(X_clean)
  print(f"Dimension of X_clean after: {X_clean.shape}")
  # Create a new feature 'Advanced_Age' based on the 'Age' column
  X_clean['Advanced_Age'] = (X_clean['Age'] >= 65).astype(int)

  # Add the ordinal columns from apply_ordinal_encoding to numeric_cols for scaling
  ordinal_cols = [col for col in ordinal_mappings.keys() if col in X_clean.columns]
  new_numeric_cols = numeric_cols + ordinal_cols + ['Advanced_Age', 'Cancer_Severity', 'Treatment_Effectiveness', 'Early_Detection_Treatment']

  scaling_true = 1
  X_clean, stats_pre = df_num_imputation(X_clean, new_numeric_cols, scaling_true)
  print(f"Dimension of X_clean after: {X_clean.shape}")
  X_train_processed = create_dummies(X_clean, categorical_cols)
  print(f"Dimension of X_clean after: {X_train_processed.shape}")
  X_train_processed.reset_index(drop=True, inplace=True)
  print(f"Dimension of X_clean after: {X_train_processed.shape}")
  X_train_processed.drop(columns='ID', inplace=True)

  return X_train_processed, y_train_processed, mode_train, stats_pre, encoders

# FUNCTIONS FOR PREPROCESSING FOR TEST SET

In [276]:
def testdf_categorical_imputation(df, values_to_imput_cat, mode_train):

  df_new = df.copy()
  categorical_columns = df_new.select_dtypes(include=['object', 'category']).columns

  # Replacing strange values of columns in values_to_imput_cat with the mode stored in mode_train dictionary
  for column, value_to_imput in values_to_imput_cat.items():
    if column in mode_train: # Verifying if the column exists in mode_train as a key
      df_new.loc[df_new[column] == values_to_imput_cat[column], column] = mode_train[column] # Replacing strange value with the mode of the column

  # Replacing missing values with the mode stored in mode_train dictionary
  for column in categorical_columns:
    if column in mode_train: # Verifying if the column exists in mode_train as a key
      df_new[column] = df_new[column].fillna(mode_train[column]) # Filling NaNs with the mode

  return df_new

In [277]:
# Using the encoders dictionary coming from apply_ordinal_encoding function (for originally encoding in the train set) to encode the test set

def apply_ordinal_encoding_test(df, encoders):
    df_encoded = df.copy()
    
    for col, mapping in encoders.items():
        if col in df_encoded.columns:
            df_encoded[col] = df_encoded[col].map(lambda x: mapping.get(x, -1) if pd.notna(x) else -1)
    
    return df_encoded

In [278]:
#Added the argument 'scaling_true' to scale the variables using the median if needed (0 = no scaling, 1 = scaling).

def testdf_numeric_imputation(df, stats_pre, numeric_cols, scaling_true = 1):
  df_new = df.copy()

# Transform columns into float type
  for column in numeric_cols:
    if column in df_new.columns:
      df_new[column] = pd.to_numeric(df_new[column], errors='coerce')

# Remove outliers using 'Winsorization'
  for column in numeric_cols:

      # Retrieving values from stats_pre dictionary. In case they do not exist, variables will get 'None' value
      lower_bound = stats_pre[column].get('lower_bound', None)
      upper_bound = stats_pre[column].get('upper_bound', None)
      median = stats_pre[column].get('median', None)

      if lower_bound is not None and upper_bound is not None:
        # Clipping datapoint outside of the range
        df_new[column] = df_new[column].clip(lower=lower_bound, upper=upper_bound)

      if median is not None:
        # Imputing missing values with the median
        df_new[column] = df_new[column].fillna(median)

  # Scaling variables using the median
  if scaling_true == 1:

    #for column in numeric_cols:
    #    if column in stats_pre:
    #        median = stats_pre[column]['median']
    #        df_new[column] = (df_new[column] - median) / median

    # BORRAR SI NO FUNCIONA
    if scaling_true == 1:
        for column in numeric_cols:
            if column in stats_pre:
                min_val = stats_pre[column].get('min', None)
                max_val = stats_pre[column].get('max', None)
                if min_val is not None and max_val is not None and max_val > min_val:
                    # Apply same formula using min/max from training
                    df_new[column] = (df_new[column] - min_val) / (max_val - min_val)

  return df_new

In [279]:
def preprocess_test_df(X, X_train_columns, values_to_imput_cat, columns_to_delete, mode_train, stats_pre, binary_cols, numeric_cols, categorical_cols, encoders):
  
  # FEATURE ENGINEERING 1:

  # Create a new feature 'Cardiometabolic_Risk' based on 'Diabetes' and 'Heart Disease History'
  X['Cardiometabolic_Risk'] = ((X['Diabetes'] == 'Yes') | (X['Heart Disease History'] == 'Yes')).astype(int)
  # Interaction between cancer stage and tumor size
  X['Cancer_Severity'] = X['Cancer Stage'].map({'Localized': 1, 'Regional': 2, 'Metastatic': 3}) * X['Tumor Size (mm)']
  # Efectiveness of treatment according to stage
  X['Treatment_Effectiveness'] = (X['Treatment Type'] != 'Palliative').astype(int) * (4 - X['Cancer Stage'].map({'Localized': 1, 'Regional': 2, 'Metastatic': 3}))
  # Early detection with appropriate treatment
  X['Early_Detection_Treatment'] = X['Early Detection'].map({'Yes': 1, 'No': 0}) * (X['Screening History'].map({'Never': 1, 'Irregular': 2, 'Regular': 3})) * (X['Treatment Type'] != 'Palliative').astype(int)

  X_clean = remove_columns(X, columns_to_delete)
  X_clean = standardize_urbal_rural(X_clean)
  X_clean = testdf_categorical_imputation(X_clean, values_to_imput_cat, mode_train)
  X_clean = convert_into_bool(X_clean, binary_cols)
  X_clean = apply_ordinal_encoding_test(X_clean, encoders) # Apply the encoders dictionary to the test set
  X_clean = create_age_column(X_clean) # Null values in Age because missing DoB should be imputed with the median next.

  # FEATURE ENGINEERING:

  # Create a new feature 'Advanced_Age' based on the 'Age' column
  X_clean['Advanced_Age'] = (X_clean['Age'] >= 65).astype(int)
  
  # Add the ordinal columns from apply_ordinal_encoding_test to numeric_cols list for scaling
  ordinal_cols = list(encoders.keys())
  new_numeric_cols = numeric_cols + ordinal_cols + ['Advanced_Age', 'Cancer_Severity', 'Treatment_Effectiveness', 'Early_Detection_Treatment']

  scaling_true = 1 # No scaling in the test set

  X_clean = testdf_numeric_imputation(X_clean, stats_pre, new_numeric_cols, scaling_true)
  X_test_processed = create_dummies(X_clean, categorical_cols)
  X_test_processed.reset_index(drop=True, inplace=True)
  X_test_processed = X_test_processed[X_train_columns]

  return X_test_processed

# PREPROCESSING PARAMETERS

In [280]:
columns_to_delete = ['Transfusion History', 'Marital Status', 'Smoking History', 'Diabetes History']

values_to_imput_cat = {
        'Healthcare Access': '?',
        'Gender': 'P'
    }

binary_cols = [
        'Heart Disease History', 'Inflammatory Bowel Disease',
        'Diabetes', 'Alcohol Consumption', 'Early Detection',
        'Family History', 'Genetic Mutation'
    ]

numeric_cols = [
        'Healthcare Costs', 'Incidence Rate per 100K', 'Mortality Rate per 100K',
        'Tumor Size (mm)', 'Age'
    ]

categorical_cols = [
        'Country', 'Gender',
        'Insurance Costs', 'Insurance Status','Non Smoker', 'Treatment Type', 'Urban or Rural'
]

ordinal_mappings = {
    'Cancer Stage': ['Localized', 'Regional', 'Metastatic'],
    'Diet Risk': ['Low', 'Moderate', 'High'],
    'Healthcare Access': ['Low', 'Moderate', 'High'],
    'Obesity BMI': ['Normal', 'Overweight', 'Obese'],
    'Physical Activity': ['Low', 'Moderate', 'High'],
    'Screening History': ['Never', 'Irregular', 'Regular']
}

mode_train = {}

stats_pre = {}

# PREPROCESSING TRAIN SET

In [281]:
cancer_df = pd.read_csv('https://raw.githubusercontent.com/gascalero/DM_II_project/4a73ca4928f2b95f960cd9b9f44c4700244ed553/data/raw/patient_train_data.csv',
                        encoding='UTF-8',
                        index_col=0,
                        sep=',',
                        on_bad_lines='skip',
                        quoting=3)
cancer_df.head(1)

,Alcohol Consumption,Cancer Stage,Country,Date of Birth,Diabetes,Diabetes History,Diet Risk,Early Detection,Family History,Gender,...,Non Smoker,Obesity BMI,Physical Activity,Screening History,Smoking History,Transfusion History,Treatment Type,Tumor Size (mm),Urban or Rural,Survival Prediction
ID,,,,,,,,,,,,,,,,,,,,,
1,No,Localized,UK,29-01-1966,No,No,Moderate,No,No,M,...,Yes,Overweight,Low,Regular,No,-,Chemotherapy,33.0,Urban,Yes


In [282]:
X_train, X_test, y_train, y_test = split_sets(cancer_df)

Dimension of X_train: (60028, 30)
Dimension of X_test: (15007, 30)
Dimension of y_train: (60028,)
Dimension of y_test: (15007,)


In [283]:
X_train_processed, y_train_processed, mode_train, stats_pre, encoders = preprocess_train_df(X_train, y_train, columns_to_delete, values_to_imput_cat, binary_cols, ordinal_mappings, numeric_cols, categorical_cols)

Dimension of X_clean after: (60028, 30)
Dimension of X_clean after: (60028, 30)
Dimension of X_clean after: (59180, 31)
Dimension of X_clean after: (59180, 31)
Dimension of X_clean after: (59180, 31)
Dimension of X_clean after: (59180, 31)
Dimension of X_clean after: (59180, 31)
Dimension of X_clean after: (59180, 32)
Dimension of X_clean after: (59180, 49)
Dimension of X_clean after: (59180, 49)


In [284]:
X_val_processed = preprocess_test_df(X_test, X_train_processed.columns, values_to_imput_cat, columns_to_delete, mode_train, stats_pre, binary_cols, numeric_cols, categorical_cols, encoders)

y_val_processed = y_test.copy()

# MODEL EVALUATION AND SELECTION (USING CROSS-VALIDATION)

In [285]:
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import f1_score, make_scorer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
import matplotlib.pyplot as plt
from xgboost import XGBClassifier
from sklearn.base import clone

In [286]:
def evaluate_models_with_cv(X, y, models_dict, cv, scoring='f1'):
    cv_splitter = StratifiedKFold(n_splits=cv, shuffle=True, random_state=42)
    f1_macro_scorer = make_scorer(f1_score, average='weighted')
    
    results = []
    
    for name, model in models_dict.items():
        print(f"Evaluating {name}...")
        cv_scores = cross_val_score(
            model, X, y, 
            cv=cv_splitter,
            scoring=f1_macro_scorer,
            n_jobs=-1
        )
        
        # Calculate training scores for each fold
        train_scores = []
        for train_idx, val_idx in cv_splitter.split(X, y):
            X_train_fold, X_val_fold = X.iloc[train_idx], X.iloc[val_idx]
            y_train_fold, y_val_fold = y.iloc[train_idx], y.iloc[val_idx]
            
            # Train the model on the training fold
            # Use clone to avoid fitting the original model
            model_clone = clone(model)
            model_clone.fit(X_train_fold, y_train_fold)
            
            # Evaluate on the training fold
            # Use the same model to predict on the training fold
            y_train_pred = model_clone.predict(X_train_fold)
            train_f1 = f1_score(y_train_fold, y_train_pred, average='weighted')
            train_scores.append(train_f1)
        
        # Calculate mean and std of training scores
        train_scores = np.array(train_scores)
        
        results.append({
            'Model': name,
            'Mean F1 Weighted (Val)': cv_scores.mean(),
            'Std F1 Weighted (Val)': cv_scores.std(),
            'Min F1 Weighted (Val)': cv_scores.min(),
            'Max F1 Weighted (Val)': cv_scores.max(),
            'Mean F1 Weighted (Train)': train_scores.mean(),
            'Std F1 Weighted (Train)': train_scores.std(),
            'Overfit Gap': train_scores.mean() - cv_scores.mean(),
            'Val Scores': cv_scores,
            'Train Scores': train_scores
        })
    
    results_df = pd.DataFrame(results)
    results_df = results_df.sort_values('Mean F1 Weighted (Val)', ascending=False)
    
    return results_df

In [294]:
# Define initial models with default or basic parameters
models = {
    'LogisticRegression': LogisticRegression(class_weight='balanced', max_iter=10000, random_state=42),
    'RandomForest': RandomForestClassifier(class_weight='balanced', random_state=42),
    'GradientBoosting': GradientBoostingClassifier(random_state=42),
    'KNeighbors': KNeighborsClassifier(weights='distance'),
    'XGBoost': XGBClassifier(eval_metric='aucpr', use_label_encoder=False, random_state=42)
}

In [293]:
# Evaluate models with cross-validation
cv = 10
cv_results = evaluate_models_with_cv(X_train_processed, y_train_processed, models, cv)

# Show results
cv_results[['Model', 'Mean F1 Weighted (Train)', 'Std F1 Weighted (Train)', 'Mean F1 Weighted (Val)', 'Std F1 Weighted (Val)' ,'Overfit Gap']]

Evaluating XGBoost...


,Model,Mean F1 Weighted (Train),Std F1 Weighted (Train),Mean F1 Weighted (Val),Std F1 Weighted (Val),Overfit Gap
0,XGBoost,0.686715,0.005322,0.506772,0.00489,0.179943


In [289]:
def evaluate_models_with_cv_selected(X, y, models_dict, cv, selected_features=None, scoring='f1'):
    
    if selected_features is not None:
        X = X[selected_features]
    
    cv_splitter = StratifiedKFold(n_splits=cv, shuffle=True, random_state=42)
    f1_macro_scorer = make_scorer(f1_score, average='weighted')
    
    results = []
    
    for name, model in models_dict.items():
        print(f"Evaluating {name}...")
        cv_scores = cross_val_score(
            model, X, y, 
            cv=cv_splitter,
            scoring=f1_macro_scorer,
            n_jobs=-1
        )
        
        # Calculate training scores for each fold
        train_scores = []
        for train_idx, val_idx in cv_splitter.split(X, y):
            X_train_fold, X_val_fold = X.iloc[train_idx], X.iloc[val_idx]
            y_train_fold, y_val_fold = y.iloc[train_idx], y.iloc[val_idx]
            
            # Train the model on the training fold
            model_clone = clone(model)
            model_clone.fit(X_train_fold, y_train_fold)
            
            # Evaluate on the training fold
            y_train_pred = model_clone.predict(X_train_fold)
            train_f1 = f1_score(y_train_fold, y_train_pred, average='weighted')
            train_scores.append(train_f1)
        
        train_scores = np.array(train_scores)
        
        results.append({
            'Model': name,
            'Mean F1 Weighted (Val)': cv_scores.mean(),
            'Std F1 Weighted (Val)': cv_scores.std(),
            'Min F1 Weighted (Val)': cv_scores.min(),
            'Max F1 Weighted (Val)': cv_scores.max(),
            'Mean F1 Weighted (Train)': train_scores.mean(),
            'Std F1 Weighted (Train)': train_scores.std(),
            'Overfit Gap': train_scores.mean() - cv_scores.mean(),
            'Val Scores': cv_scores,
            'Train Scores': train_scores
        })
    
    results_df = pd.DataFrame(results)
    results_df = results_df.sort_values('Mean F1 Weighted (Val)', ascending=False)
    
    return results_df

In [ ]:
# Crear un DataFrame con solo las características seleccionadas
X_simplified = X_train_processed[selected_features].copy()

# Definir modelos a evaluar
models_2 = {
    'LogisticRegression': LogisticRegression(C=0.1, class_weight='balanced', max_iter=10000, random_state=42),
    'RandomForest': RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=42),
    'GradientBoosting': GradientBoostingClassifier(n_estimators=100, random_state=42),
    'XGBoost': XGBClassifier(eval_metric='logloss', use_label_encoder=False, random_state=42)
}

X_train_simplified = X_train_processed[selected_features]
cv_results_simplified = evaluate_models_with_cv_selected(
    X_train_simplified, 
    y_train_processed, 
    models_2, 
    cv=10
)

# Comparar resultados con todas las características vs. características seleccionadas
print("Resultados con características seleccionadas:")
cv_results_simplified[['Model', 'Mean F1 Weighted (Train)', 'Std F1 Weighted (Train)', 'Mean F1 MWeighted (Val)', 'Std F1 Weighted (Val)' ,'Overfit Gap']]

Evaluating LogisticRegression...
Evaluating RandomForest...


KeyboardInterrupt: 

# GRIDSEARCH

In [247]:
X_train_processed_simplified = X_train_processed[selected_features].copy()
X_val_processed_simplified = X_val_processed[selected_features].copy()


param_grid_simplified = {
    'C': [0.01, 0.1, 1, 10, 20, 50, 100],
    'penalty': ['l2'],
    'solver': ['liblinear', 'lbfgs', 'saga'],
    'class_weight': ['balanced'],
    'max_iter': [1000000]
}

log_reg = LogisticRegression(random_state=42)

# Definir qué métrica optimizar (F1-score es bueno para datos desbalanceados)
scorer = make_scorer(f1_score, average='weighted')

# Configurar GridSearchCV
grid_search = GridSearchCV(
    estimator=log_reg,
    param_grid=param_grid_simplified,
    scoring=scorer,
    cv=10,  # Validación cruzada de 5 folds
    n_jobs=-1,  # Usar todos los núcleos disponibles
    verbose=2,   # Mostrar progreso
    return_train_score=True
)

# Ajustar GridSearch a los datos
grid_search.fit(X_train_processed_simplified, y_train_processed)

# Ver los mejores parámetros
print("Mejores parámetros encontrados:")
print(grid_search.best_params_)
print(f"Mejor puntuación F1: {grid_search.best_score_:.4f}")

# Crear modelo con los mejores parámetros
best_log_reg = LogisticRegression(**grid_search.best_params_, random_state=42)

# Entrenar con todos los datos
best_log_reg.fit(X_train_processed_simplified, y_train_processed)

# Evaluar en conjunto de prueba
y_pred_best = best_log_reg.predict(X_val_processed_simplified)
print("\nInforme de clasificación con los mejores parámetros:")
print(classification_report(y_val_processed, y_pred_best))

Fitting 10 folds for each of 21 candidates, totalling 210 fits
Mejores parámetros encontrados:
{'C': 0.01, 'class_weight': 'balanced', 'max_iter': 1000000, 'penalty': 'l2', 'solver': 'saga'}
Mejor puntuación F1: 0.5213

Informe de clasificación con los mejores parámetros:
              precision    recall  f1-score   support

           0       0.40      0.44      0.42      5976
           1       0.60      0.57      0.58      9031

    accuracy                           0.52     15007
   macro avg       0.50      0.50      0.50     15007
weighted avg       0.52      0.52      0.52     15007



# KAGGLE DF (NO TOCAR HASTA CUANDO HAYAS SELECCIONADO MODELO Y PARÁMETROS CON MEJORAS SIGNIFICATIVAS!!!)

In [250]:
test_cancer_df = pd.read_csv('https://raw.githubusercontent.com/gascalero/DM_II_project/refs/heads/master/data/raw/patient_test_data.csv',
                         encoding='UTF-8',
                         index_col=0,
                         sep=',',
                         on_bad_lines='skip',
                         quoting=3)

test_cancer_df.head(1)

,Alcohol Consumption,Cancer Stage,Country,Date of Birth,Diabetes,Diabetes History,Diet Risk,Early Detection,Family History,Gender,...,Mortality Rate per 100K,Non Smoker,Obesity BMI,Physical Activity,Screening History,Smoking History,Transfusion History,Treatment Type,Tumor Size (mm),Urban or Rural
ID,,,,,,,,,,,,,,,,,,,,,
75036,Yes,Localized,UK,17-11-1947,No,No,Low,Yes,No,M,...,5.0,Yes,Overweight,Low,Regular,No,-,Combination,69.0,Urban


In [251]:
X_test_processed = preprocess_test_df(test_cancer_df, X_train_processed.columns, values_to_imput_cat, columns_to_delete, mode_train, stats_pre, binary_cols, numeric_cols, categorical_cols, encoders)

In [258]:
X_train_processed_simplified = X_train_processed[selected_features].copy()
X_test_processed_simplified = X_test_processed[selected_features].copy()

logistic_model = LogisticRegression(C= 0.01, class_weight= 'balanced', max_iter= 1000000, penalty= 'l2', solver= 'saga')

logistic_model.fit(X_train_processed_simplified, y_train_processed)
y_pred_logistic = logistic_model.predict(X_test_processed_simplified)

In [259]:
df_kaggle = pd.DataFrame(y_pred_logistic, index=test_cancer_df.index)

df_kaggle.replace({0: 'No', 1: 'Yes'}, inplace = True)

df_kaggle.columns = ['Survival Prediction']

df_kaggle.value_counts()

Survival Prediction
Yes                    42466
No                     32534
Name: count, dtype: int64

In [213]:
df_kaggle.to_csv('DT_Group05_Version12.csv')